# Environment Setup and Data Loading

- This notebook uses PySpark in Databricks to analyse the ICLR and OpenAlex datasets.

- Both datasets are loaded using Spark DataFrames from the required file paths. 
Initial checks such as `printSchema(), count(), and display()` are used to understand 
the structure of the data, including column types, nested fields, and overall size.

- This follows the approach used in the lab practicals, where data is explored first 
before applying transformations. All processing is done using PySpark functions, 
ensuring the solution meets the coursework requirement of using distributed 
(big data) tools rather than non-parallel approaches like pandas.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
ICLR_PATH = "/Volumes/workspace/default/big_data/iclr26v1.parquet"
OPENALEX_PATH = "/Volumes/workspace/default/big_data/openalex.json"

df_iclr = spark.read.parquet(ICLR_PATH)
df_openalex = spark.read.json(OPENALEX_PATH)

In [0]:
display(dbutils.fs.ls("/Volumes/workspace/default/big_data/"))

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/big_data/iclr26v1.parquet,iclr26v1.parquet,48993373,1777663277000
dbfs:/Volumes/workspace/default/big_data/openalex.json,openalex.json,31234421,1777663245000


In [0]:
df_iclr.printSchema()
df_openalex.printSchema()

root
 |-- year: long (nullable = true)
 |-- id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- abstract: string (nullable = true)
 |-- authors: string (nullable = true)
 |-- author_ids: string (nullable = true)
 |-- decision: string (nullable = true)
 |-- scores: array (nullable = true)
 |    |-- element: long (containsNull = true)
 |-- keywords: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- labels: string (nullable = true)

root
 |-- apc_list: struct (nullable = true)
 |    |-- currency: string (nullable = true)
 |    |-- value: long (nullable = true)
 |    |-- value_usd: long (nullable = true)
 |-- apc_paid: struct (nullable = true)
 |    |-- currency: string (nullable = true)
 |    |-- value: long (nullable = true)
 |    |-- value_usd: long (nullable = true)
 |-- authorships: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- affiliations: array (nullable = true)
 |    |    |    |-- element: struc

In [0]:
print("ICLR rows:", df_iclr.count())
print("OpenAlex rows:", df_openalex.count())

ICLR rows: 55906
OpenAlex rows: 2245


In [0]:
display(df_iclr.limit(5))
display(df_openalex.limit(5))

year,id,title,abstract,authors,author_ids,decision,scores,keywords,labels
2017,B1-Hhnslg,Prototypical Networks for Few-shot Learning,"A recent approach to few-shot classification called matching networks has demonstrated the benefits of coupling metric learning with a training procedure that mimics test. This approach relies on a complicated fine-tuning procedure and an attention scheme that forms a distribution over all points in the support set, scaling poorly with its size. We propose a more streamlined approach, prototypical networks, that learns a metric space in which few-shot classification can be performed by computing Euclidean distances to prototype representations of each class, rather than individual points. Our method is competitive with state-of-the-art one-shot classification approaches while being much simpler and more scalable with the size of the support set. We empirically demonstrate the performance of our approach on the Omniglot and mini-ImageNet datasets. We further demonstrate that a similar idea can be used for zero-shot learning, where each class is described by a set of attributes, and achieve state-of-the-art results on the Caltech UCSD bird dataset.","Jake Snell, Kevin Swersky, Richard Zemel",,Reject,"List(6, 4, 5)","List(deep learning, transfer learning)",transfer learning
2017,B1-q5Pqxl,Machine Comprehension Using Match-LSTM and Answer Pointer,"Machine comprehension of text is an important problem in natural language processing. A recently released dataset, the Stanford Question Answering Dataset (SQuAD), offers a large number of real questions and their answers created by humans through crowdsourcing. SQuAD provides a challenging testbed for evaluating machine comprehension algorithms, partly because compared with previous datasets, in SQuAD the answers do not come from a small set of candidate answers and they have variable lengths. We propose an end-to-end neural architecture for the task. The architecture is based on match-LSTM, a model we proposed previously for textual entailment, and Pointer Net, a sequence-to-sequence model proposed by Vinyals et al. (2015) to constrain the output tokens to be from the input sequences. We propose two ways of using Pointer Net for our tasks. Our experiments show that both of our two models substantially outperform the best results obtained by Rajpurkar et al. (2016) using logistic regression and manually crafted features. Besides, our boundary model also achieves the best performance on the MSMARCO dataset (Nguyen et al. 2016).","Shuohang Wang, Jing Jiang",,Accept (Poster),"List(6, 6, 7)","List(natural language processing, deep learning)",language models
2017,B16Jem9xe,Learning in Implicit Generative Models,"Generative adversarial networks (GANs) provide an algorithmic framework for constructing generative models with several appealing properties: they do not require a likelihood function to be specified, only a generating procedure; they provide samples that are sharp and compelling; and they allow us to harness our knowledge of building highly accurate neural network classifiers. Here, we develop our understanding of GANs with the aim of forming a rich view of this growing area of machine learning---to build connections to the diverse set of statistical thinking on this topic, of which much can be gained by a mutual exchange of ideas. We frame GANs within the wider landscape of algorithms for learning in implicit generative models---models that only specify a stochastic procedure with which to generate data---and relate these ideas to modelling problems in related fields, such as econometrics and approximate Bayesian computation. We develop likelihood-free inference methods and highlight hypothesis testing as a principle for learning in implicit generative models, using which we are able to derive the objective function used by GANs, and many other related objectives. The testing viewpoint directs our focus to the general problem of density ratio estimat

apc_list,apc_paid,authorships,awards,best_oa_location,biblio,citation_normalized_percentile,cited_by_count,cited_by_percentile_year,concepts,contains_iclr,content_urls,corresponding_author_ids,corresponding_institution_ids,countries_distinct_count,counts_by_year,created_date,display_name,doi,funders,fwci,has_content,has_fulltext,id,ids,indexed_in,institutions,institutions_distinct_count,is_paratext,is_retracted,is_xpac,keywords,language,locations,locations_count,mesh,open_access,primary_location,primary_topic,publication_date,publication_year,referenced_works,referenced_works_count,related_works,relevance_score,sustainable_development_goals,title,topics,type,updated_date
null,null,"List(List(List(List(List(https://openalex.org/I1291425158), Google,,,,,)), List(Irina Higgins, https://openalex.org/A5018830387, https://orcid.org/0000-0002-1890-2091), first, List(US), List(List(US, Google (United States), https://openalex.org/I1291425158, List(https://openalex.org/I1291425158, https://openalex.org/I4210128969), https://ror.org/00njsd438, company)), true, List(Google,,,,,), Irina Higgins), List(List(List(List(https://openalex.org/I1291425158), Google,,,,,)), List(Löıc Matthey, https://openalex.org/A5113031232, null), middle, List(US), List(List(US, Google (United States), https://openalex.org/I1291425158, List(https://openalex.org/I1291425158, https://openalex.org/I4210128969), https://ror.org/00njsd438, company)), false, List(Google,,,,,), Löıc Matthey), List(List(List(List(https://openalex.org/I1291425158), Google,,,,,)), List(Arka Pal, https://openalex.org/A5045751957, null), middle, List(US), List(List(US, Google (United States), https://openalex.org/I1291425158, List(https://openalex.org/I1291425158, https://openalex.org/I4210128969), https://ror.org/00njsd438, company)), false, List(Google,,,,,), Arka Pal), List(List(List(List(https://openalex.org/I1291425158), Google,,,,,)), List(Christopher Burgess, https://openalex.org/A5109247980, null), middle, List(US), List(List(US, Google (United States), https://openalex.org/I1291425158, List(https://openalex.org/I1291425158, https://openalex.org/I4210128969), https://ror.org/00njsd438, company)), false, List(Google,,,,,), Christopher Burgess), List(List(List(List(https://openalex.org/I1291425158), Google,,,,,)), List(Xavier Glorot, https://openalex.org/A5067483843, https://orcid.org/0000-0002-6729-2189), middle, List(US), List(List(US, Google (United States), https://openalex.org/I1291425158, List(https://openalex.org/I1291425158, https://openalex.org/I4210128969), https://ror.org/00njsd438, company)), false, List(Google,,,,,), Xavier Glorot), List(List(List(List(https://openalex.org/I1291425158), Google,,,,,)), List(Matthew Botvinick, https://openalex.org/A5083771180, https://orcid.org/0000-0001-7758-6896), middle, List(US), List(List(US, Google (United States), https://openalex.org/I1291425158, List(https://openalex.org/I1291425158, https://openalex.org/I4210128969), https://ror.org/00njsd438, company)), false, List(Google,,,,,), Matthew Botvinick), List(List(List(List(https://openalex.org/I1291425158), Google,,,,,)), List(Shakir Mohamed, https://openalex.org/A5084530286, https://orcid.org/0000-0002-1184-5776), middle, List(US), List(List(US, Google (United States), https://openalex.org/I1291425158, List(https://openalex.org/I1291425158, https://openalex.org/I4210128969), https://ror.org/00njsd438, company)), false, List(Google,,,,,), Shakir Mohamed), List(List(List(List(https://openalex.org/I1291425158), Google,,,,,)), List(Alexander Lerchner, https://openalex.org/A5035836682, null), last, List(US), List(List(US, Google (United States), https://openalex.org/I1291425158, List(https://openalex.org/I1291425158, https://openalex.org/I4210128969), https://ror.org/00njsd438, company)), false, List(Google,,,,,), Alexander Lerchner))",List(),null,"List(null, null, null, null)","List(true, true, 0.99942029)",3085,"List(100, 96)","List(List(Autoencoder, https://openalex.org/C101738243, 3, 

# Data Cleaning and Preparation

Only the preprocessing steps required for the analysis are applied to the datasets.

- First, the decision column is standardised into consistent categories (accepted, rejected, and withdrawn), as the original data contains different formats for similar outcomes. This is necessary for Questions 1 and 2.

- Next, a mean score is calculated from the reviewer scores array. This is required for Questions 3 and 6, where numerical comparisons of paper quality are needed.

- Finally, paper titles are cleaned (lowercased and stripped of punctuation) to support dataset matching in Questions 5 and 6.

- 
No additional cleaning steps are applied to the basic question answers. The `title_clean `column is used only for extension analyses, ensuring the basic Q5 join remains consistent with the provided model answers as required by the assignment brief.

In [0]:
df_iclr_clean = (
    df_iclr
    .withColumn("decision_lower", F.lower(F.trim(F.col("decision"))))
    .withColumn(
        "decision_clean",
        F.when(F.col("decision_lower").contains("withdraw"), "withdrawn")
        .when(
            (F.col("decision_lower").contains("accept")) |
            (F.col("decision_lower").contains("poster")) |
            (F.col("decision_lower").contains("oral")) |
            (F.col("decision_lower").contains("spotlight")) |
            (F.col("decision_lower").contains("workshop")),
            "accepted"
        )
        .when(F.col("decision_lower").contains("reject"), "rejected")
        .otherwise("other")
    )
    .withColumn(
        "mean_score",
        F.when(
            F.size("scores") > 0,
            F.expr("aggregate(scores, 0D, (acc, x) -> acc + x) / size(scores)")
        )
    )
    .withColumn(
        "title_clean",
        F.lower(F.trim(F.regexp_replace(F.col("title"), "[^a-zA-Z0-9 ]", " ")))
    )
)

df_openalex_clean = df_openalex.withColumn(
    "title_clean",
    F.lower(F.trim(F.regexp_replace(F.col("title"), "[^a-zA-Z0-9 ]", " ")))
)

display(df_iclr_clean.limit(5))

year,id,title,abstract,authors,author_ids,decision,scores,keywords,labels,decision_lower,decision_clean,mean_score,title_clean
2017,B1-Hhnslg,Prototypical Networks for Few-shot Learning,"A recent approach to few-shot classification called matching networks has demonstrated the benefits of coupling metric learning with a training procedure that mimics test. This approach relies on a complicated fine-tuning procedure and an attention scheme that forms a distribution over all points in the support set, scaling poorly with its size. We propose a more streamlined approach, prototypical networks, that learns a metric space in which few-shot classification can be performed by computing Euclidean distances to prototype representations of each class, rather than individual points. Our method is competitive with state-of-the-art one-shot classification approaches while being much simpler and more scalable with the size of the support set. We empirically demonstrate the performance of our approach on the Omniglot and mini-ImageNet datasets. We further demonstrate that a similar idea can be used for zero-shot learning, where each class is described by a set of attributes, and achieve state-of-the-art results on the Caltech UCSD bird dataset.","Jake Snell, Kevin Swersky, Richard Zemel",,Reject,"List(6, 4, 5)","List(deep learning, transfer learning)",transfer learning,reject,rejected,5.0,prototypical networks for few shot learning
2017,B1-q5Pqxl,Machine Comprehension Using Match-LSTM and Answer Pointer,"Machine comprehension of text is an important problem in natural language processing. A recently released dataset, the Stanford Question Answering Dataset (SQuAD), offers a large number of real questions and their answers created by humans through crowdsourcing. SQuAD provides a challenging testbed for evaluating machine comprehension algorithms, partly because compared with previous datasets, in SQuAD the answers do not come from a small set of candidate answers and they have variable lengths. We propose an end-to-end neural architecture for the task. The architecture is based on match-LSTM, a model we proposed previously for textual entailment, and Pointer Net, a sequence-to-sequence model proposed by Vinyals et al. (2015) to constrain the output tokens to be from the input sequences. We propose two ways of using Pointer Net for our tasks. Our experiments show that both of our two models substantially outperform the best results obtained by Rajpurkar et al. (2016) using logistic regression and manually crafted features. Besides, our boundary model also achieves the best performance on the MSMARCO dataset (Nguyen et al. 2016).","Shuohang Wang, Jing Jiang",,Accept (Poster),"List(6, 6, 7)","List(natural language processing, deep learning)",language models,accept (poster),accepted,6.333333333333333,machine comprehension using match lstm and answer pointer
2017,B16Jem9xe,Learning in Implicit Generative Models,"Generative adversarial networks (GANs) provide an algorithmic framework for constructing generative models with several appealing properties: they do not require a likelihood function to be specified, only a generating procedure; they provide samples that are sharp and compelling; and they allow us to harness our knowledge of building highly accurate neural network classifiers. Here, we develop our understanding of GANs with the aim of forming a rich view of this growing area of machine learning---to build connections to the diverse set of statistical thinking on this topic, of which much can be gained by a mutual exchange of ideas. We frame GANs within the wider landscape of algorithms for learning in implicit generative models---models that only specify a stochastic procedure with which to generate data---and relate these ideas to modelling problems in related fields, such as econometrics and approximate Bayesian computation. We develop likelihood-free inference methods and highlight hypothesis testing as a principle for learning in imp

In [0]:
display(
    df_iclr_clean
    .groupBy("decision", "decision_clean")
    .count()
)

decision,decision_clean,count
Accept: notable-top-25%,accepted,280
Accept (Poster),accepted,6173
Accept (Talk),accepted,48
Reject,rejected,17096
Accept (Spotlight),accepted,775
Invite to Workshop Track,accepted,136
Desk rejected,rejected,194
,other,19673
Accept (Oral),accepted,383
Accept: poster,accepted,1202


In [0]:
display(
    df_iclr_clean
    .select("scores", "mean_score")
)

scores,mean_score
"List(6, 4, 5)",5.0
"List(6, 6, 7)",6.333333333333333
"List(8, 7, 6)",7.0
"List(6, 5, 6)",5.666666666666667
"List(7, 9, 5)",7.0
"List(3, 4, 4, 4)",3.75
"List(6, 4, 3)",4.333333333333333
"List(7, 8, 7)",7.333333333333333
"List(4, 3, 3)",3.3333333333333335
"List(7, 8, 8)",7.666666666666667


# Question 1: Number of Accepted, Rejected and Withdrawn Papers


In [0]:
q1_result = (
    df_iclr_clean
    .filter(F.col("decision_clean").isin("accepted", "rejected", "withdrawn",))
    .groupBy("decision_clean")
    .agg(F.count("*").alias("paper_count"))
    .orderBy("decision_clean")
)

display(q1_result)

decision_clean,paper_count
accepted,11349
rejected,17290
withdrawn,7594


# 1.1 Extension: Z-Score Analysis of Annual Acceptance Rates — Identifying Statistically Anomalous Years

In [0]:
# Compute acceptance rate per year
acceptance_by_year = (
    df_iclr_clean
    .filter(F.col("decision_clean").isin("accepted", "rejected", "withdrawn"))
    .groupBy("year")
    .agg(
        F.count("*").alias("total_submissions"),
        F.sum(
            F.when(F.col("decision_clean") == "accepted", 1).otherwise(0)
        ).alias("accepted_count")
    )
    .withColumn(
        "acceptance_rate",
        F.round(F.col("accepted_count") / F.col("total_submissions") * 100, 2)
    )
)

# Z-Score: how many standard deviations is each year from the mean acceptance rate
window_all = Window.rowsBetween(
    Window.unboundedPreceding, Window.unboundedFollowing
)

q1_extension_zscore = (
    acceptance_by_year
    .withColumn("mean_rate", F.avg("acceptance_rate").over(window_all))
    .withColumn("std_rate", F.stddev("acceptance_rate").over(window_all))
    .withColumn(
        "z_score",
        F.round(
            (F.col("acceptance_rate") - F.col("mean_rate")) / F.col("std_rate"),
            3
        )
    )
    .withColumn(
        "year_type",
        F.when(F.col("z_score") >  1.5, "Unusually Lenient")
         .when(F.col("z_score") < -1.5, "Unusually Selective")
         .otherwise("Normal Range")
    )
    .select(
        "year", "total_submissions", "accepted_count",
        "acceptance_rate", "z_score", "year_type"
    )
    .orderBy("year")
)

display(q1_extension_zscore)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


year,total_submissions,accepted_count,acceptance_rate,z_score,year_type
2017,489,245,50.1,1.53,Unusually Lenient
2018,1012,425,42.0,0.897,Normal Range
2019,1569,502,31.99,0.115,Normal Range
2020,2593,687,26.49,-0.315,Normal Range
2021,3009,859,28.55,-0.154,Normal Range
2022,3422,1094,31.97,0.114,Normal Range
2023,4955,1573,31.75,0.097,Normal Range
2024,7401,2261,30.55,0.003,Normal Range
2025,11663,3703,31.75,0.097,Normal Range
2026,120,0,0.0,-2.384,Unusually Selective


# Question 2: Accepted Papers with At Least One Low Score

In [0]:
q2_result = (
    df_iclr_clean
    .filter(
        (F.col("decision_clean") == "accepted") &
        (F.array_min(F.col("scores")) < 5)
    )
)

q2_count = q2_result.count()
print("Number of accepted papers with at least one score below 5:", q2_count)

Number of accepted papers with at least one score below 5: 1596


# 2.1 Extension: Min-Max Normalisation of Mean Scores — Classifying Low-Score Accepted Papers by Severity of Reviewer Disagreement

In [0]:
# Min-Max Normalisation of mean scores for low-score accepted papers

# Step 1: Get global min and max
bounds = (
    df_iclr_clean
    .filter(
        (F.col("decision_clean") == "accepted") &
        (F.array_min(F.col("scores")) < 5) &
        F.col("mean_score").isNotNull()
    )
    .agg(
        F.min("mean_score").alias("min_score"),
        F.max("mean_score").alias("max_score")
    )
    .collect()[0]
)

min_s = bounds["min_score"]
max_s = bounds["max_score"]

# Step 2: Apply formula: normalised = (x - min) / (max - min)
q2_extension = (
    df_iclr_clean
    .filter(
        (F.col("decision_clean") == "accepted") &
        (F.array_min(F.col("scores")) < 5) &
        F.col("mean_score").isNotNull()
    )
    .withColumn(
        "normalised_score",
        F.round(
            (F.col("mean_score") - F.lit(min_s)) / F.lit(max_s - min_s),
            3
        )
    )
    .withColumn(
        "score_tier",
        F.when(F.col("normalised_score") >= 0.7, "Upper Tier")
         .when(F.col("normalised_score") >= 0.4, "Mid Tier")
         .otherwise("Lower Tier")
    )
    .groupBy("score_tier")
    .agg(
        F.count("*").alias("paper_count"),
        F.round(F.avg("mean_score"), 2).alias("avg_raw_score"),
        F.round(F.avg("normalised_score"), 3).alias("avg_normalised_score")
    )
    .orderBy(F.col("avg_normalised_score").desc())
)

display(q2_extension)

score_tier,paper_count,avg_raw_score,avg_normalised_score
Upper Tier,117,6.75,0.752
Mid Tier,1392,5.65,0.566
Lower Tier,87,4.31,0.338


# Question 3: Top 10 Papers with Highest Mean Score


In [0]:
q3_result = (
    df_iclr_clean
    .filter(F.col("mean_score").isNotNull())
    .select(
        "id",
        "mean_score",
        "year"
    )
    .orderBy(F.col("mean_score").desc())
    .limit(10)
)

display(q3_result)

id,mean_score,year
u1cQYxRI1H,10.0,2025
Sy8gdB9xx,9.666666666666666,2017
6Mxhg9PtDE,9.5,2025
88nT0j5jAn,9.333333333333334,2023
LyJi5ugyJx,9.2,2025
lTt4KjHSsyl,9.0,2023
b-ny3x071E5,9.0,2022
aN4Jf6Cx69,9.0,2024
DJSZGGZYVi,9.0,2025
WCRQFlji2q,9.0,2025


# 3.1 Extension: Score Variance Analysis — Verifying Whether Top 10 Mean Scores Reflect Genuine Reviewer Consensus or Outlier Inflation

In [0]:
# Score Variance (Standard Deviation) for Top 10 Papers

top10_ids = [row["id"] for row in q3_result.select("id").collect()]

q3_extension = (
    df_iclr_clean
    .filter(F.col("id").isin(top10_ids))
    .select(
        "id", "year", "mean_score", "decision_clean",
        F.explode("scores").alias("score")
    )
    .groupBy("id", "year", "mean_score", "decision_clean")
    .agg(
        F.round(F.stddev("score"), 3).alias("score_stddev"),
        F.count("score").alias("num_reviewers")
    )
    .orderBy(F.col("mean_score").desc())
)

display(q3_extension)

id,year,mean_score,decision_clean,score_stddev,num_reviewers
u1cQYxRI1H,2025,10.0,accepted,0.0,4
Sy8gdB9xx,2017,9.666666666666666,accepted,0.577,3
6Mxhg9PtDE,2025,9.5,accepted,1.0,4
88nT0j5jAn,2023,9.333333333333334,accepted,1.155,3
LyJi5ugyJx,2025,9.2,accepted,1.095,5
aN4Jf6Cx69,2024,9.0,accepted,1.155,4
DJSZGGZYVi,2025,9.0,accepted,1.095,6
b-ny3x071E5,2022,9.0,accepted,1.155,4
lTt4KjHSsyl,2023,9.0,accepted,1.155,4
WCRQFlji2q,2025,9.0,accepted,1.155,4


# Question 4: Rapidly Increasing Keywords


In [0]:
keywords_df = (
    df_iclr_clean
    .select("year", F.explode("keywords").alias("keyword"))
    .withColumn("keyword_clean", F.lower(F.trim(F.col("keyword"))))
    .filter(F.col("keyword_clean").isNotNull())
    .filter(F.col("keyword_clean") != "")
)

keyword_counts = (
    keywords_df
    .groupBy("keyword_clean", "year")
    .agg(F.count("*").alias("keyword_count"))
)

keyword_growth = (
    keyword_counts.alias("current")
    .join(
        keyword_counts.alias("next"),
        (F.col("current.keyword_clean") == F.col("next.keyword_clean")) &
        (F.col("next.year") == F.col("current.year") + 1),
        "inner"
    )
    .select(
        F.col("current.keyword_clean").alias("keyword"),
        F.col("current.year").alias("year"),
        F.col("current.keyword_count").alias("low_count"),
        F.col("next.keyword_count").alias("high_count"),
        F.col("next.year").alias("following_year")
    )
)

q4_result = (
    keyword_growth
    .filter((F.col("low_count") < 5) & (F.col("high_count") > 50))
    .orderBy(F.col("high_count").desc())
    .limit(10)
)

display(q4_result)

keyword,year,low_count,high_count,following_year
test-time scaling,2025,3,87,2026
llm,2023,4,76,2024
spatial reasoning,2025,3,57,2026
large reasoning models,2025,1,57,2026


# 4.1 Extension: Full Growth Trajectory and Year-on-Year Growth Rate Analysis of Identified Emerging Keywords

In [0]:
# Get identified keywords from Q4 basic result
q4_extension_keywords = [
    row["keyword"] for row in q4_result.select("keyword").collect()
]

# Self-contained: recompute clean keyword counts directly
kw_clean = (
    df_iclr_clean
    .select("year", F.explode("keywords").alias("kw"))
    .withColumn("keyword_clean", F.lower(F.trim(F.col("kw"))))
    .filter(F.col("keyword_clean").isNotNull())
    .filter(F.col("keyword_clean") != "")
    .groupBy("keyword_clean", "year")
    .agg(F.count("*").alias("keyword_count"))
)

# Window lag: get previous year count for each keyword
window_kw = Window.partitionBy("keyword_clean").orderBy("year")

q4_extension = (
    kw_clean
    .filter(F.col("keyword_clean").isin(q4_extension_keywords))
    .withColumn(
        "prev_count",
        F.lag("keyword_count", 1).over(window_kw)
    )
    .withColumn(
        "growth_rate_pct",
        F.when(
            (F.col("prev_count").isNotNull()) & (F.col("prev_count") > 0),
            F.round(
                (F.col("keyword_count") - F.col("prev_count"))
                / F.col("prev_count") * 100, 1
            )
        ).otherwise(F.lit(None))
    )
    .select(
        "keyword_clean", "year",
        "keyword_count", "prev_count", "growth_rate_pct"
    )
    .orderBy("keyword_clean", "year")
)

display(q4_extension)

keyword_clean,year,keyword_count,prev_count,growth_rate_pct
large reasoning models,2025,1,null,null
large reasoning models,2026,57,1,5600.0
llm,2023,4,null,null
llm,2024,76,4,1800.0
llm,2025,350,76,360.5
llm,2026,804,350,129.7
spatial reasoning,2024,1,null,null
spatial reasoning,2025,3,1,200.0
spatial reasoning,2026,57,3,1800.0
test-time scaling,2025,3,null,null


# Question 5: ICLR Papers with OpenAlex Information


In [0]:
q5_join = (
    df_iclr.alias("iclr")
    .join(
        df_openalex.alias("oa"),
        F.col("iclr.title") == F.col("oa.title"),
        "inner"
    )
)

q5_result = (
    q5_join
    .groupBy(F.col("iclr.year").alias("year"))
    .agg(F.count("*").alias("matched_papers"))
    .orderBy("year")
)

display(q5_result)

year,matched_papers
2017,154
2018,309
2019,199
2020,433
2021,423
2022,52
2023,19
2024,1
2025,2
2026,1


# 5.1 Extension: Impact of Title Normalisation on Match Count — Quantifying Additional Matches Recovered Through Cleaned Title Join

In [0]:
# Extension: Impact of Title Normalisation on Match Count

q5_cleaned_result = (
    df_iclr_clean.alias("iclr")
    .join(
        df_openalex_clean.alias("oa"),
        F.col("iclr.title_clean") == F.col("oa.title_clean"),
        "inner"
    )
    .groupBy(F.col("iclr.year").alias("year"))
    .agg(F.count("*").alias("cleaned_title_matches"))
)

q5_exact_result = (
    q5_result
    .withColumnRenamed("matched_papers", "exact_title_matches")
)

q5_extension = (
    q5_exact_result
    .join(q5_cleaned_result, "year", "outer")
    .withColumn(
        "additional_matches",
        F.col("cleaned_title_matches") - F.col("exact_title_matches")
    )
    .orderBy("year")
)

display(q5_extension)

year,exact_title_matches,cleaned_title_matches,additional_matches
2017,154,174,20
2018,309,353,44
2019,199,256,57
2020,433,471,38
2021,423,458,35
2022,52,66,14
2023,19,20,1
2024,1,1,0
2025,2,3,1
2026,1,1,0


# Question 6: Low Score but Highly Cited Papers


In [0]:
q6_join = (
    df_iclr_clean.alias("iclr")
    .join(
        df_openalex.alias("oa"),
        F.col("iclr.title") == F.col("oa.title"),
        "inner"
    )
)


In [0]:
p95 = q6_join.approxQuantile("cited_by_count", [0.95], 0.01)[0]
print("95th percentile:", p95)

95th percentile: 299.0


In [0]:
q6_result = (
    q6_join
    .filter(
        (F.col("iclr.mean_score") < 5) &
        (F.col("oa.citation_normalized_percentile.value") >= 0.95)
    )
    .select(
        F.col("iclr.id").alias("id"),
        F.col("iclr.year").alias("year"),
    )
    .orderBy("year", "id")
)
display(q6_result)

id,year
S19eAF9ee,2017
BkPrDFgR-,2018
H1bM1fZCW,2018
HJWGdbbCW,2018
HkmaTz-0W,2018
Hy-lXyDWG,2018
HJeq43AqF7,2019
B1esx6EYvr,2020
ByexElSYDr,2020
Byg5flHFDr,2020


# 6.1 Extension: Pearson Correlation and Score Band Analysis — Quantifying the Relationship Between Reviewer Scores and Long-Term Citation Impact

In [0]:
# Extension: Does reviewer score predict citation impact?

# Step 1: Clean deduplicated base
q6_base = (
    q6_join
    .filter(F.col("iclr.mean_score").isNotNull())
    .filter(F.col("oa.cited_by_count").isNotNull())
    .select(
        F.col("iclr.id").alias("id"),
        F.col("iclr.mean_score").alias("mean_score"),
        F.col("oa.cited_by_count").alias("cited_by_count")
    )
    .groupBy("id", "mean_score")
    .agg(F.max("cited_by_count").alias("cited_by_count"))
)

# Step 2: Pearson correlation
pearson = q6_base.agg(
    F.round(F.corr("mean_score", "cited_by_count"), 4)
    .alias("pearson_correlation")
)
display(pearson)

# Step 3: Score band analysis
q6_extension = (
    q6_base
    .withColumn("score_band",
        F.when(F.col("mean_score") < 4, "Low (below 4)")
         .when(F.col("mean_score") < 6, "Medium (4–6)")
         .when(F.col("mean_score") < 8, "High (6–8)")
         .otherwise("Very High (8+)")
    )
    .groupBy("score_band")
    .agg(
        F.count("*").alias("paper_count"),
        F.round(F.avg("cited_by_count"), 1).alias("avg_citations"),
        F.round(F.max("cited_by_count"), 0).alias("max_citations")
    )
    .orderBy("score_band")
)
display(q6_extension)

pearson_correlation
0.0472


score_band,paper_count,avg_citations,max_citations
High (6–8),980,84.4,5915
Low (below 4),45,23.9,128
Medium (4–6),416,79.1,3471
Very High (8+),43,151.0,1866
